# Relevant Python 3.8 and 3.7 Changes — Advanced Problems with Solutions

This notebook expands the supplied lesson into advanced, executable exercises.

**Target runtime:** Python 3.8+  
**Main topics:**

1. Positional-only parameters (`/`)
2. Self-documenting f-strings (`{expr=}`)
3. `as_integer_ratio()` across numeric types
4. `functools.lru_cache` enhancements and cache design
5. `math.dist`
6. `namedtuple` defaults, `_field_defaults`, and `_asdict()`
7. Reversing dictionary views
8. `continue` in `finally`
9. `SyntaxWarning` for `is` with literals
10. Cross-topic capstone problems

> Best-practice principle: new syntax is useful only when it makes an API safer,
> clearer, more testable, or less error-prone.

## How to use this notebook

For each problem:

- Read the requirements.
- Try the **Starter** cell first.
- Compare your work with the **Solution**.
- Run the verification cell.
- Read the best-practice notes and edge cases.

All expected failures are caught so that **Run All** can complete successfully.

In [1]:
import math
import sys
import warnings
from collections import namedtuple
from decimal import Decimal
from fractions import Fraction
from functools import lru_cache, wraps
from inspect import signature

print(f"{sys.version=}")
assert sys.version_info >= (3, 8), "This notebook requires Python 3.8 or newer."

sys.version='3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]'


# 1. Positional-Only Parameters

Parameters before `/` can only be supplied positionally. Parameters after `*`
can only be supplied by keyword. Used together, they create precise public APIs.

## Problem 1 — Design a stable pricing API

Create `final_price(subtotal, tax_rate, /, discount=0, *, precision=2)`.

Requirements:

- `subtotal` and `tax_rate` are positional-only because their names are
  implementation details.
- `precision` is keyword-only because a positional value would be unclear.
- Reject negative `subtotal`.
- Require `0 <= discount <= subtotal`.
- Return `(subtotal - discount) * (1 + tax_rate)`, rounded to `precision`.
- Use `Decimal` internally to avoid binary floating-point surprises.

In [2]:
# Starter
# def final_price(subtotal, tax_rate, /, discount=0, *, precision=2):
#     ...

### Solution

In [3]:
def final_price(subtotal, tax_rate, /, discount=0, *, precision=2):
    subtotal = Decimal(str(subtotal))
    tax_rate = Decimal(str(tax_rate))
    discount = Decimal(str(discount))

    if subtotal < 0:
        raise ValueError("subtotal must be non-negative")
    if not Decimal("0") <= discount <= subtotal:
        raise ValueError("discount must be between 0 and subtotal")
    if not isinstance(precision, int) or precision < 0:
        raise ValueError("precision must be a non-negative integer")

    amount = (subtotal - discount) * (Decimal("1") + tax_rate)
    quantum = Decimal("1").scaleb(-precision)
    return amount.quantize(quantum)


assert final_price(100, 0.20, 10, precision=2) == Decimal("108.00")
assert final_price("19.99", "0.20", precision=2) == Decimal("23.99")

try:
    final_price(subtotal=100, tax_rate=0.20)
except TypeError as exc:
    print("Expected positional-only failure:", exc)

try:
    final_price(100, 0.20, precision=-1)
except ValueError as exc:
    print("Expected validation failure:", exc)

Expected positional-only failure: final_price() got some positional-only arguments passed as keyword arguments: 'subtotal, tax_rate'
Expected validation failure: precision must be a non-negative integer


**Best practices**

- Use `/` when parameter names should not become part of the public contract.
- Use `*` for configuration-like arguments that are clearest when named.
- Do not make every parameter positional-only; reserve it for stable,
  intentionally constrained APIs.

## Problem 2 — Allow keyword names that match positional-only parameters

Implement:

```python
def record_event(event_type, payload, /, **metadata):
    ...
```

The caller must be allowed to include a metadata key named `"event_type"`
without colliding with the positional-only parameter.

In [4]:
# Starter
# def record_event(event_type, payload, /, **metadata):
#     ...

### Solution

In [5]:
def record_event(event_type, payload, /, **metadata):
    if not isinstance(event_type, str) or not event_type:
        raise ValueError("event_type must be a non-empty string")
    if not isinstance(payload, dict):
        raise TypeError("payload must be a dictionary")

    return {
        "event_type": event_type,
        "payload": dict(payload),
        "metadata": dict(metadata),
    }


event = record_event(
    "LOGIN",
    {"user_id": 42},
    event_type="security-audit",
    source="web",
)

assert event["event_type"] == "LOGIN"
assert event["metadata"]["event_type"] == "security-audit"
event

{'event_type': 'LOGIN',
 'payload': {'user_id': 42},
 'metadata': {'event_type': 'security-audit', 'source': 'web'}}

This is a subtle but powerful use of `/`: the name `event_type` remains available
inside `**metadata` because the formal parameter cannot be bound by keyword.

## Problem 3 — Decorate a constrained API without breaking its behavior

Write a `trace_calls` decorator for a function containing both `/` and `*`.
Preserve metadata with `functools.wraps`, log arguments, and confirm that the
original call restrictions are still enforced.

In [6]:
# Starter
# def trace_calls(func):
#     ...

### Solution

In [7]:
def trace_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}: {args=}, {kwargs=}")
        result = func(*args, **kwargs)
        print(f"{result=}")
        return result
    return wrapper


@trace_calls
def clamp(value, lower, upper, /, *, inclusive=True):
    if lower > upper:
        raise ValueError("lower must not exceed upper")

    if inclusive:
        return min(max(value, lower), upper)

    if not lower < value < upper:
        raise ValueError("value must be strictly inside the interval")
    return value


assert clamp(15, 0, 10, inclusive=True) == 10
print("Displayed signature:", signature(clamp))

try:
    clamp(value=5, lower=0, upper=10)
except TypeError as exc:
    print("Restrictions still enforced:", exc)

calling clamp: args=(15, 0, 10), kwargs={'inclusive': True}
result=10
Displayed signature: (value, lower, upper, /, *, inclusive=True)
calling clamp: args=(), kwargs={'value': 5, 'lower': 0, 'upper': 10}
Restrictions still enforced: clamp() got some positional-only arguments passed as keyword arguments: 'value, lower, upper'


`wraps` sets `__wrapped__`, allowing tools such as `inspect.signature` to expose
the original signature even though the wrapper itself accepts `*args, **kwargs`.

## Problem 4 — Refactor an ambiguous function signature

The following API is easy to misuse:

```python
def export(data, format="json", compress=False, destination=None):
    ...
```

Refactor it so that:

- `data` is positional-only.
- `format`, `compress`, and `destination` are keyword-only.
- Only `"json"` and `"csv"` are accepted.
- The function returns a configuration dictionary.

In [8]:
def export(data, /, *, format="json", compress=False, destination=None):
    if format not in {"json", "csv"}:
        raise ValueError("format must be 'json' or 'csv'")
    if not isinstance(compress, bool):
        raise TypeError("compress must be bool")

    return {
        "data": data,
        "format": format,
        "compress": compress,
        "destination": destination,
    }


config = export([{"id": 1}], format="csv", compress=True, destination="out.csv.gz")
assert config["format"] == "csv"
assert config["compress"] is True

try:
    export([1, 2, 3], "csv", True, "out.csv")
except TypeError as exc:
    print("Ambiguous positional configuration rejected:", exc)

Ambiguous positional configuration rejected: export() takes 1 positional argument but 4 were given


# 2. Self-Documenting f-Strings

Python 3.8 introduced the debugging form `{expression=}`. It displays both the
source expression and its value.

## Problem 5 — Produce a compact numeric diagnostic line

Given `samples`, compute count, mean, minimum, maximum, and range. Print one line
using self-documenting expressions and format the mean to three decimal places.

In [9]:
samples = [3.5, 8.25, 2.0, 9.75, 6.5]

count = len(samples)
mean = sum(samples) / count
minimum = min(samples)
maximum = max(samples)
spread = maximum - minimum

diagnostic = (
    f"{count=}, {mean=:.3f}, {minimum=:.2f}, "
    f"{maximum=:.2f}, {spread=:.2f}"
)
print(diagnostic)

assert "count=5" in diagnostic
assert "mean=6.000" in diagnostic

count=5, mean=6.000, minimum=2.00, maximum=9.75, spread=7.75


## Problem 6 — Debug expressions, conversions, and format specifications

Show the difference between default `repr`, explicit `!s`, and numeric
formatting when using the `=` debugging form.

In [10]:
name = "Ada\nLovelace"
ratio = 1 / 3

default_debug = f"{name=}"
string_debug = f"{name=!s}"
formatted_debug = f"{ratio=:.2%}"

print(default_debug)
print(string_debug)
print(formatted_debug)

assert default_debug == "name='Ada\\nLovelace'"
assert string_debug == "name=Ada\nLovelace"
assert formatted_debug == "ratio=33.33%"

name='Ada\nLovelace'
name=Ada
Lovelace
ratio=33.33%


The debugging form defaults to `repr` unless a format specification causes
normal formatting behavior. Use `!s`, `!r`, or `!a` deliberately when logs have
specific readability or escaping requirements.

## Problem 7 — Prevent accidental repeated side effects in debug output

A function call inside an f-string is still evaluated. Demonstrate the unsafe
pattern, then rewrite it so the side-effecting function is called exactly once
and the result is logged clearly.

In [11]:
calls = 0

def next_ticket():
    global calls
    calls += 1
    return f"T-{calls:03d}"


# Unsafe when copied twice: each expression invokes the function.
unsafe = f"{next_ticket()=}, {next_ticket()=}"
print(unsafe)
assert calls == 2

# Best practice: evaluate once, assign, then log.
calls = 0
ticket = next_ticket()
safe = f"{ticket=}"
print(safe)
assert calls == 1
assert safe == "ticket='T-001'"

next_ticket()='T-001', next_ticket()='T-002'
ticket='T-001'


Self-documenting f-strings are debugging tools, not a substitute for structured
logging. Avoid secrets, large payloads, unstable `repr` output, and expensive
expressions in production logs.

# 3. `as_integer_ratio()` and Numeric Duck Typing

## Problem 8 — Convert supported numeric objects to exact `Fraction` values

Implement `exact_fraction(value, *, allow_bool=False)`.

Requirements:

- Accept objects implementing `as_integer_ratio()`.
- Reject booleans by default, even though `bool` supports the method.
- Validate that the returned denominator is positive.
- Raise a clear `TypeError` for unsupported inputs.

In [12]:
# Starter
# def exact_fraction(value, *, allow_bool=False):
#     ...

### Solution

In [13]:
def exact_fraction(value, *, allow_bool=False):
    if isinstance(value, bool) and not allow_bool:
        raise TypeError("bool is not accepted as a numeric measurement")

    method = getattr(value, "as_integer_ratio", None)
    if method is None:
        raise TypeError(
            f"{type(value).__name__} does not implement as_integer_ratio()"
        )

    numerator, denominator = method()

    if denominator <= 0:
        raise ValueError("denominator must be positive")

    return Fraction(numerator, denominator)


examples = [
    12,
    0.5,
    Decimal("0.125"),
    Fraction(7, 9),
]

converted = [exact_fraction(value) for value in examples]
print(converted)

assert converted == [
    Fraction(12, 1),
    Fraction(1, 2),
    Fraction(1, 8),
    Fraction(7, 9),
]

try:
    exact_fraction(True)
except TypeError as exc:
    print("Expected bool policy:", exc)

[Fraction(12, 1), Fraction(1, 2), Fraction(1, 8), Fraction(7, 9)]
Expected bool policy: bool is not accepted as a numeric measurement


## Problem 9 — Explain why `0.1` and `Decimal("0.1")` differ

Use exact ratios to calculate the precise error introduced by the binary float
literal `0.1`.

In [14]:
float_fraction = exact_fraction(0.1)
decimal_fraction = exact_fraction(Decimal("0.1"))
error = float_fraction - decimal_fraction

print(f"{float_fraction=}")
print(f"{decimal_fraction=}")
print(f"{error=}")
print(f"{float(error)=:.20e}")

assert decimal_fraction == Fraction(1, 10)
assert error != 0

float_fraction=Fraction(3602879701896397, 36028797018963968)
decimal_fraction=Fraction(1, 10)
error=Fraction(1, 180143985094819840)
float(error)=5.55111512312578301027e-18


`float.as_integer_ratio()` reveals the exact stored binary floating-point value.
It does not "simplify" the value to what a human probably intended.

## Problem 10 — Build a safe ratio extractor for finite values

Handle normal values, positive infinity, negative infinity, NaN, and unsupported
objects without allowing exceptions to escape.

In [15]:
def safe_integer_ratio(value):
    try:
        numerator, denominator = value.as_integer_ratio()
    except AttributeError:
        return {"ok": False, "error": "unsupported type"}
    except (OverflowError, ValueError) as exc:
        return {"ok": False, "error": str(exc)}
    else:
        return {
            "ok": True,
            "numerator": numerator,
            "denominator": denominator,
        }


cases = [3, 0.25, float("inf"), float("-inf"), float("nan"), "3/4"]
results = [safe_integer_ratio(value) for value in cases]

for value, result in zip(cases, results):
    print(f"{value=!r} -> {result}")

assert results[0]["ok"] is True
assert results[2]["ok"] is False
assert results[-1]["error"] == "unsupported type"

value=3 -> {'ok': True, 'numerator': 3, 'denominator': 1}
value=0.25 -> {'ok': True, 'numerator': 1, 'denominator': 4}
value=inf -> {'ok': False, 'error': 'cannot convert Infinity to integer ratio'}
value=-inf -> {'ok': False, 'error': 'cannot convert Infinity to integer ratio'}
value=nan -> {'ok': False, 'error': 'cannot convert NaN to integer ratio'}
value='3/4' -> {'ok': False, 'error': 'unsupported type'}


## Problem 11 — Normalize heterogeneous numeric weights

Write a function that accepts `int`, `float`, `Decimal`, and `Fraction` weights,
converts them exactly, and returns normalized fractions summing to one.

In [16]:
def normalize_weights(values):
    fractions = [exact_fraction(value) for value in values]

    if not fractions:
        raise ValueError("at least one weight is required")
    if any(value < 0 for value in fractions):
        raise ValueError("weights must be non-negative")

    total = sum(fractions, start=Fraction(0, 1))
    if total == 0:
        raise ValueError("at least one weight must be positive")

    normalized = [value / total for value in fractions]
    assert sum(normalized, start=Fraction(0, 1)) == 1
    return normalized


weights = normalize_weights([1, Decimal("0.5"), Fraction(1, 4), 0.25])
print(weights)
assert weights == [
    Fraction(1, 2),
    Fraction(1, 4),
    Fraction(1, 8),
    Fraction(1, 8),
]

[Fraction(1, 2), Fraction(1, 4), Fraction(1, 8), Fraction(1, 8)]


# 4. `lru_cache`: Correctness, Keys, and Eviction

## Problem 12 — Cache an edit-distance solver

Implement Levenshtein distance with a cached recursive helper. Return both the
distance and cache statistics.

In [17]:
def edit_distance(left, right):
    @lru_cache
    def solve(i, j):
        if i == len(left):
            return len(right) - j
        if j == len(right):
            return len(left) - i

        substitution_cost = 0 if left[i] == right[j] else 1

        return min(
            1 + solve(i + 1, j),                 # delete
            1 + solve(i, j + 1),                 # insert
            substitution_cost + solve(i + 1, j + 1),  # replace/match
        )

    distance = solve(0, 0)
    return distance, solve.cache_info()


distance, info = edit_distance("kitten", "sitting")
print(f"{distance=}, {info=}")

assert distance == 3
assert info.hits > 0
assert info.currsize <= (len("kitten") + 1) * (len("sitting") + 1)

distance=3, info=CacheInfo(hits=71, misses=56, maxsize=128, currsize=56)


The cached helper uses only hashable integers as keys. Keeping the cache local
also prevents unrelated calls from accumulating forever in a process-wide cache.

## Problem 13 — Observe bounded LRU eviction

Predict and verify the cache statistics after this access sequence:

`1, 2, 3, 1, 4, 2`

Use `maxsize=3`.

In [18]:
@lru_cache(maxsize=3)
def square(value):
    print(f"computing {value=}")
    return value * value


sequence = [1, 2, 3, 1, 4, 2]
outputs = [square(value) for value in sequence]
info = square.cache_info()

print(f"{outputs=}")
print(f"{info=}")

assert outputs == [1, 4, 9, 1, 16, 4]
assert info.hits == 1
assert info.misses == 5
assert info.currsize == 3

computing value=1
computing value=2
computing value=3
computing value=4
computing value=2
outputs=[1, 4, 9, 1, 16, 4]
info=CacheInfo(hits=1, misses=5, maxsize=3, currsize=3)


Calling `1` refreshes its recency. Adding `4` therefore evicts `2`, so the final
call with `2` is another miss.

## Problem 14 — Cache a function that receives a list

`lru_cache` requires hashable arguments. Build a public wrapper that accepts any
iterable, converts it to a tuple, and delegates to a cached internal function.

In [19]:
@lru_cache
def _moving_average_cached(values, window):
    if window <= 0:
        raise ValueError("window must be positive")
    if window > len(values):
        return ()

    return tuple(
        sum(values[index:index + window]) / window
        for index in range(len(values) - window + 1)
    )


def moving_average(values, window):
    return _moving_average_cached(tuple(values), window)


data = [1, 2, 3, 4, 5]
first = moving_average(data, 3)
second = moving_average(list(data), 3)
info = _moving_average_cached.cache_info()

print(f"{first=}, {info=}")
assert first == (2.0, 3.0, 4.0)
assert second == first
assert info.hits == 1

first=(2.0, 3.0, 4.0), info=CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


Convert mutable inputs at the API boundary. Do not mutate an object after using
an equivalent hashable representation as a cache key and assume the cache knows.

## Problem 15 — Canonicalize keyword arguments before caching

Calls that differ only in keyword order should share one cache entry. Implement a
small decorator that sorts keyword items before they reach the cached layer.

Assumption: all arguments and keyword values are hashable.

In [20]:
def canonical_lru_cache(func):
    @lru_cache
    def cached(args, sorted_kwargs):
        return func(*args, **dict(sorted_kwargs))

    @wraps(func)
    def wrapper(*args, **kwargs):
        key = tuple(sorted(kwargs.items()))
        return cached(args, key)

    wrapper.cache_info = cached.cache_info
    wrapper.cache_clear = cached.cache_clear
    return wrapper


calls = 0

@canonical_lru_cache
def build_query(table, **filters):
    global calls
    calls += 1
    clauses = " AND ".join(
        f"{name}={value!r}" for name, value in sorted(filters.items())
    )
    return f"SELECT * FROM {table} WHERE {clauses}"


query_1 = build_query("users", active=True, role="admin")
query_2 = build_query("users", role="admin", active=True)

print(query_1)
print(build_query.cache_info())

assert query_1 == query_2
assert calls == 1
assert build_query.cache_info().hits == 1

SELECT * FROM users WHERE active=True AND role='admin'
CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)


For production-grade canonicalization, define policies for nested mutable values,
duplicate semantic forms, default values, and type distinctions. Cache keys are
part of correctness, not merely performance.

## Problem 16 — Decide whether `typed=True` is needed

Explore how cache key behavior can depend on argument types. Then create a cache
that explicitly distinguishes types.

In [21]:
typed_calls = 0

@lru_cache(maxsize=None, typed=True)
def describe_number(value):
    global typed_calls
    typed_calls += 1
    return type(value).__name__, value


results = [
    describe_number(1),
    describe_number(1.0),
    describe_number(True),
    describe_number(Decimal("1")),
]

print(results)
print(describe_number.cache_info())

assert results[0][0] == "int"
assert results[1][0] == "float"
assert results[2][0] == "bool"
assert results[3][0] == "Decimal"
assert typed_calls == 4

[('int', 1), ('float', 1.0), ('bool', True), ('Decimal', Decimal('1'))]
CacheInfo(hits=0, misses=4, maxsize=None, currsize=4)


Use `typed=True` only when type differences are semantically meaningful. It may
increase cache size and does not recursively distinguish the types of items
inside containers.

# 5. `math.dist` for N-Dimensional Geometry

## Problem 17 — Find the nearest point in arbitrary dimensions

Implement `nearest_point(origin, points)` using `math.dist`.

Requirements:

- Support any dimension.
- Reject an empty candidate collection.
- Reject dimension mismatches with a useful error.
- Return `(point, distance)`.

In [22]:
def nearest_point(origin, points):
    origin = tuple(origin)
    points = [tuple(point) for point in points]

    if not points:
        raise ValueError("points must not be empty")

    dimension = len(origin)
    if any(len(point) != dimension for point in points):
        raise ValueError("all points must have the same dimension as origin")

    point = min(points, key=lambda candidate: math.dist(origin, candidate))
    return point, math.dist(origin, point)


origin = (0, 0, 0)
candidates = [(5, 0, 0), (1, 1, 1), (0, 0, 2)]
point, distance = nearest_point(origin, candidates)

print(f"{point=}, {distance=:.6f}")
assert point == (1, 1, 1)
assert math.isclose(distance, math.sqrt(3))

point=(1, 1, 1), distance=1.732051


## Problem 18 — Implement weighted Euclidean distance via scaling

For positive weights `w_i`, weighted Euclidean distance is:

\[
\sqrt{\sum_i w_i (x_i-y_i)^2}
\]

Transform each coordinate by `sqrt(w_i)` and then call `math.dist`.

In [23]:
def weighted_distance(left, right, weights):
    left = tuple(left)
    right = tuple(right)
    weights = tuple(weights)

    if not (len(left) == len(right) == len(weights)):
        raise ValueError("left, right, and weights must have equal lengths")
    if any(weight <= 0 for weight in weights):
        raise ValueError("weights must be positive")

    scaled_left = tuple(
        coordinate * math.sqrt(weight)
        for coordinate, weight in zip(left, weights)
    )
    scaled_right = tuple(
        coordinate * math.sqrt(weight)
        for coordinate, weight in zip(right, weights)
    )
    return math.dist(scaled_left, scaled_right)


result = weighted_distance((1, 2), (4, 6), (4, 0.25))
manual = math.sqrt(4 * (1 - 4) ** 2 + 0.25 * (2 - 6) ** 2)

print(f"{result=}, {manual=}")
assert math.isclose(result, manual)
assert math.isclose(result, math.sqrt(40))

result=6.324555320336759, manual=6.324555320336759


## Problem 19 — Build a cached distance matrix

Create an immutable tuple-of-tuples distance matrix. Cache the calculation so
equivalent tuple inputs reuse the result.

In [24]:
@lru_cache
def distance_matrix(points):
    points = tuple(tuple(point) for point in points)

    if len({len(point) for point in points}) > 1:
        raise ValueError("all points must have equal dimensions")

    return tuple(
        tuple(math.dist(left, right) for right in points)
        for left in points
    )


points = ((0, 0), (3, 4), (6, 8))
matrix_1 = distance_matrix(points)
matrix_2 = distance_matrix(tuple(points))

for row in matrix_1:
    print(row)

assert matrix_1 == matrix_2
assert matrix_1[0][1] == 5.0
assert matrix_1[0][2] == 10.0
assert distance_matrix.cache_info().hits == 1

(0.0, 5.0, 10.0)
(5.0, 0.0, 5.0)
(10.0, 5.0, 0.0)


# 6. `namedtuple` Changes and Design

## Problem 20 — Reason about right-aligned defaults

Create `Connection(host, port, timeout, secure)` where only `host` is required,
and defaults are:

- `port=443`
- `timeout=5.0`
- `secure=True`

Inspect `_field_defaults`.

In [25]:
Connection = namedtuple(
    "Connection",
    "host port timeout secure",
    defaults=(443, 5.0, True),
)

connection = Connection("example.com")

print(connection)
print(Connection._field_defaults)

assert connection == Connection(
    host="example.com",
    port=443,
    timeout=5.0,
    secure=True,
)
assert Connection._field_defaults == {
    "port": 443,
    "timeout": 5.0,
    "secure": True,
}

Connection(host='example.com', port=443, timeout=5.0, secure=True)
{'port': 443, 'timeout': 5.0, 'secure': True}


Defaults are applied to the rightmost fields. A required field cannot appear
after a defaulted field, mirroring ordinary function parameter rules.

## Problem 21 — Diagnose and repair mutable defaults

First demonstrate why this definition is dangerous:

```python
BadRecord = namedtuple("BadRecord", "name tags", defaults=([],))
```

Then provide a safe factory that gives every record an independent list.

In [26]:
BadRecord = namedtuple("BadRecord", "name tags", defaults=([],))

bad_a = BadRecord("A")
bad_b = BadRecord("B")
bad_a.tags.append("shared")

print(f"{bad_a=}")
print(f"{bad_b=}")
assert bad_a.tags is bad_b.tags


Record = namedtuple("Record", "name tags", defaults=(None,))

def make_record(name, tags=None):
    independent_tags = [] if tags is None else list(tags)
    return Record(name, independent_tags)


good_a = make_record("A")
good_b = make_record("B")
good_a.tags.append("independent")

print(f"{good_a=}")
print(f"{good_b=}")

assert good_a.tags is not good_b.tags
assert good_b.tags == []

bad_a=BadRecord(name='A', tags=['shared'])
bad_b=BadRecord(name='B', tags=['shared'])
good_a=Record(name='A', tags=['independent'])
good_b=Record(name='B', tags=[])


A `namedtuple` field can still reference a mutable object. Tuple immutability
prevents field reassignment, not mutation of the object stored in a field.

## Problem 22 — Serialize with `_asdict()` and preserve field order

Create a `Job` record, convert it to a normal dictionary, enrich it, and verify
that key order follows field order.

In [27]:
Job = namedtuple(
    "Job",
    "job_id owner priority status",
    defaults=("normal", "queued"),
)

job = Job(101, "Ada")
payload = job._asdict()
payload["display"] = f"{job.job_id}: {job.owner} [{job.status}]"

print(f"{type(payload)=}")
print(payload)

assert type(payload) is dict
assert list(payload) == [
    "job_id",
    "owner",
    "priority",
    "status",
    "display",
]

type(payload)=<class 'dict'>
{'job_id': 101, 'owner': 'Ada', 'priority': 'normal', 'status': 'queued', 'display': '101: Ada [queued]'}


## Problem 23 — Validate a named tuple through a factory

`namedtuple` does not provide field validation by default. Build a validated
factory for a `Vector(x, y, z)` record.

In [28]:
Vector = namedtuple("Vector", "x y z", defaults=(0.0, 0.0, 0.0))

def make_vector(x=0.0, y=0.0, z=0.0):
    values = (x, y, z)
    if any(isinstance(value, bool) for value in values):
        raise TypeError("boolean coordinates are not accepted")
    if not all(isinstance(value, (int, float, Decimal, Fraction)) for value in values):
        raise TypeError("coordinates must be numeric")
    return Vector(*values)


vector = make_vector(3, 4)
assert vector == Vector(3, 4, 0.0)
assert math.dist(vector, Vector()) == 5.0

try:
    make_vector(True, 0, 0)
except TypeError as exc:
    print("Expected validation failure:", exc)

Expected validation failure: boolean coordinates are not accepted


# 7. Reversing Dictionary Views

## Problem 24 — Retrieve the most recently inserted matching item

Implement `find_latest(mapping, predicate)` using reverse iteration over
`mapping.items()`. Return `(key, value)` or raise `LookupError`.

In [29]:
def find_latest(mapping, predicate):
    for key, value in reversed(mapping.items()):
        if predicate(key, value):
            return key, value
    raise LookupError("no matching item")


status_history = {
    "09:00": "queued",
    "09:15": "running",
    "09:20": "warning",
    "09:30": "running",
    "09:45": "complete",
}

latest_active = find_latest(
    status_history,
    lambda key, value: value in {"running", "warning"},
)

print(latest_active)
assert latest_active == ("09:30", "running")

('09:30', 'running')


## Problem 25 — Safely delete the last `n` inserted dictionary entries

Mutating a dictionary while iterating over one of its live views is unsafe.
Snapshot the keys first.

In [30]:
def remove_latest(mapping, count):
    if count < 0:
        raise ValueError("count must be non-negative")

    keys_to_remove = list(reversed(mapping.keys()))[:count]
    removed = []

    for key in keys_to_remove:
        removed.append((key, mapping.pop(key)))

    return removed


inventory = {"A": 10, "B": 20, "C": 30, "D": 40}
removed = remove_latest(inventory, 2)

print(f"{removed=}")
print(f"{inventory=}")

assert removed == [("D", 40), ("C", 30)]
assert inventory == {"A": 10, "B": 20}

removed=[('D', 40), ('C', 30)]
inventory={'A': 10, 'B': 20}


## Problem 26 — Traverse a dictionary in reverse without copying it

Calculate the first three values encountered in reverse insertion order without
building a reversed list of the entire mapping.

In [31]:
measurements = {f"m{index}": index ** 2 for index in range(10)}

reverse_iterator = reversed(measurements.values())
last_three = [next(reverse_iterator) for _ in range(3)]

print(last_three)
assert last_three == [81, 64, 49]

[81, 64, 49]


The reverse iterator itself is lazy. A list copy is only needed when you require
a stable snapshot, random access, or safe mutation of the original dictionary.

# 8. `continue` in `finally`: Legal but Dangerous

## Problem 27 — Show how `continue` in `finally` can suppress an exception

Trace this control flow, then implement a safe rewrite that performs cleanup
without hiding failures.

In [32]:
def unsafe_process(values):
    accepted = []
    audit = []

    for value in values:
        try:
            if value < 0:
                raise ValueError(f"negative value: {value}")
            accepted.append(value)
        finally:
            audit.append(f"cleaned:{value}")
            if value < 0:
                # Legal in Python 3.8+, but it suppresses the pending exception.
                continue

    return accepted, audit


accepted, audit = unsafe_process([1, -2, 3])
print(f"{accepted=}")
print(f"{audit=}")

assert accepted == [1, 3]
assert audit == ["cleaned:1", "cleaned:-2", "cleaned:3"]

accepted=[1, 3]
audit=['cleaned:1', 'cleaned:-2', 'cleaned:3']


### Safe rewrite

In [33]:
def safe_process(values):
    accepted = []
    rejected = []
    audit = []

    for value in values:
        try:
            if value < 0:
                rejected.append((value, "negative"))
                continue
            accepted.append(value)
        finally:
            # Cleanup only; no return, break, continue, or new exception.
            audit.append(f"cleaned:{value}")

    return accepted, rejected, audit


accepted, rejected, audit = safe_process([1, -2, 3])

print(f"{accepted=}")
print(f"{rejected=}")
print(f"{audit=}")

assert accepted == [1, 3]
assert rejected == [(-2, "negative")]
assert len(audit) == 3

accepted=[1, 3]
rejected=[(-2, 'negative')]
audit=['cleaned:1', 'cleaned:-2', 'cleaned:3']


Best practice: keep `finally` focused on cleanup. Control-transfer statements
inside `finally` can override `return`, `break`, `continue`, or exceptions that
were already in flight.

## Problem 28 — Prefer a context manager for cleanup

Create a tiny context manager that records acquisition and release, then use
`continue` in the loop body rather than in `finally`.

In [34]:
class TrackedResource:
    def __init__(self, name, log):
        self.name = name
        self.log = log

    def __enter__(self):
        self.log.append(f"open:{self.name}")
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.log.append(f"close:{self.name}")
        return False  # Never suppress exceptions.


log = []
processed = []

for value in [1, -1, 2]:
    with TrackedResource(str(value), log):
        if value < 0:
            continue
        processed.append(value * 10)

print(f"{processed=}")
print(f"{log=}")

assert processed == [10, 20]
assert log == [
    "open:1", "close:1",
    "open:-1", "close:-1",
    "open:2", "close:2",
]

processed=[10, 20]
log=['open:1', 'close:1', 'open:-1', 'close:-1', 'open:2', 'close:2']


# 9. Identity, Equality, and `SyntaxWarning`

## Problem 29 — Capture the warning for `is` with a literal

Use `compile()` inside `warnings.catch_warnings` so the notebook can inspect the
warning without containing invalid comparison style in ordinary code.

In [35]:
source = "value = 1\nresult = (value is 1)\n"

with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always", SyntaxWarning)
    compiled = compile(source, "<identity-demo>", "exec")

messages = [str(item.message) for item in captured]
print(messages)

assert any("Did you mean" in message and "==" in message for message in messages)

['"is" with \'int\' literal. Did you mean "=="?']


## Problem 30 — Use equality for values and identity for sentinels

Implement `lookup(mapping, key, /, *, default=MISSING)`. Distinguish a missing
key from a key whose stored value is `None`.

In [36]:
MISSING = object()

def lookup(mapping, key, /, *, default=MISSING):
    value = mapping.get(key, MISSING)

    if value is MISSING:
        if default is MISSING:
            raise KeyError(key)
        return default

    return value


data = {"present_none": None, "count": 0}

assert lookup(data, "present_none") is None
assert lookup(data, "count") == 0
assert lookup(data, "absent", default="fallback") == "fallback"

try:
    lookup(data, "absent")
except KeyError as exc:
    print("Expected missing-key failure:", exc)

Expected missing-key failure: 'absent'


`is` is appropriate for singleton sentinels such as `None`, `NotImplemented`,
`Ellipsis`, and a private `object()` instance. Use `==` for value comparison.

## Problem 31 — Demonstrate equal-but-not-identical objects

Create two distinct objects with equal values and verify the difference between
`==` and `is`.

In [37]:
left = [1, 2, 3]
right = [1, 2, 3]
alias = left

print(f"{left == right=}")
print(f"{left is right=}")
print(f"{left is alias=}")

assert left == right
assert left is not right
assert left is alias

left == right=True
left is right=False
left is alias=True


# 10. Cross-Topic Capstone Problems

## Problem 32 — Build a cached geometric event analyzer

Combine these features:

- A `namedtuple` with right-aligned defaults.
- A positional-only public argument.
- Keyword-only configuration.
- `math.dist`.
- `lru_cache` without empty parentheses.
- Self-documenting f-string diagnostics.
- `_asdict()` for output.
- Reverse dictionary traversal to find the latest matching event.

An event contains `event_id`, `point`, `weight`, and `label`.

In [38]:
Event = namedtuple(
    "Event",
    "event_id point weight label",
    defaults=(1, "unlabeled"),
)


def make_event(event_id, point, weight=1, label="unlabeled"):
    point = tuple(point)
    if not point:
        raise ValueError("point must contain at least one coordinate")
    if isinstance(weight, bool):
        raise TypeError("boolean weights are not accepted")

    exact_weight = exact_fraction(weight)
    if exact_weight < 0:
        raise ValueError("weight must be non-negative")

    return Event(event_id, point, exact_weight, label)


@lru_cache
def _distance(origin, point):
    return math.dist(origin, point)


def analyze_events(events, origin, /, *, limit=3, required_label=None):
    origin = tuple(origin)
    events = tuple(events)

    if limit < 0:
        raise ValueError("limit must be non-negative")
    if any(len(event.point) != len(origin) for event in events):
        raise ValueError("all event points must match origin dimension")

    ranked = sorted(
        events,
        key=lambda event: _distance(origin, event.point),
    )

    if required_label is not None:
        ranked = [
            event for event in ranked
            if event.label == required_label
        ]

    selected = ranked[:limit]
    output = []

    for event in selected:
        row = event._asdict()
        row["distance"] = _distance(origin, event.point)
        row["weighted_distance"] = row["distance"] * float(event.weight)
        output.append(row)

    print(f"{origin=}, {limit=}, {required_label=}, {len(output)=}")
    return output


events = (
    make_event("E1", (0, 0), Decimal("0.5"), "normal"),
    make_event("E2", (3, 4), Fraction(3, 2), "alert"),
    make_event("E3", (1, 1), 2, "alert"),
    make_event("E4", (10, 10)),
)

report = analyze_events(events, (0, 0), limit=2, required_label="alert")
report_again = analyze_events(events, (0, 0), limit=2, required_label="alert")

for row in report:
    print(row)

assert [row["event_id"] for row in report] == ["E3", "E2"]
assert math.isclose(report[0]["distance"], math.sqrt(2))
assert _distance.cache_info().hits > 0

origin=(0, 0), limit=2, required_label='alert', len(output)=2
origin=(0, 0), limit=2, required_label='alert', len(output)=2
{'event_id': 'E3', 'point': (1, 1), 'weight': Fraction(2, 1), 'label': 'alert', 'distance': 1.4142135623730951, 'weighted_distance': 2.8284271247461903}
{'event_id': 'E2', 'point': (3, 4), 'weight': Fraction(3, 2), 'label': 'alert', 'distance': 5.0, 'weighted_distance': 7.5}


### Capstone extension — Find the latest alert state

Event state history is stored in insertion order. Retrieve the latest alert
without reversing a copied list.

In [39]:
event_states = {
    "08:00": {"event_id": "E1", "label": "normal"},
    "08:10": {"event_id": "E2", "label": "alert"},
    "08:20": {"event_id": "E3", "label": "normal"},
    "08:30": {"event_id": "E4", "label": "alert"},
}

latest_alert = find_latest(
    event_states,
    lambda timestamp, state: state["label"] == "alert",
)

print(f"{latest_alert=}")
assert latest_alert == (
    "08:30",
    {"event_id": "E4", "label": "alert"},
)

latest_alert=('08:30', {'event_id': 'E4', 'label': 'alert'})


## Problem 33 — Build a memoized route scorer with exact weights

Implement a route scorer where:

- A route is a tuple of N-dimensional points.
- Segment distance uses `math.dist`.
- A numeric multiplier is normalized through `as_integer_ratio()`.
- The internal calculation is cached.
- The public route is positional-only.
- Debug output uses `{expr=}`.

In [40]:
@lru_cache
def _route_length(route):
    return sum(
        math.dist(left, right)
        for left, right in zip(route, route[1:])
    )


def score_route(route, /, *, multiplier=1):
    route = tuple(tuple(point) for point in route)

    if len(route) < 2:
        raise ValueError("route must contain at least two points")

    dimensions = {len(point) for point in route}
    if len(dimensions) != 1:
        raise ValueError("all route points must have equal dimensions")

    exact_multiplier = exact_fraction(multiplier)
    if exact_multiplier < 0:
        raise ValueError("multiplier must be non-negative")

    length = _route_length(route)
    score = length * float(exact_multiplier)

    print(f"{length=:.3f}, {exact_multiplier=}, {score=:.3f}")
    return score


route = [(0, 0), (3, 4), (6, 8)]
score_1 = score_route(route, multiplier=Decimal("1.5"))
score_2 = score_route(tuple(route), multiplier=Fraction(3, 2))

assert math.isclose(score_1, 15.0)
assert math.isclose(score_2, 15.0)
assert _route_length.cache_info().hits == 1

length=10.000, exact_multiplier=Fraction(3, 2), score=15.000
length=10.000, exact_multiplier=Fraction(3, 2), score=15.000


## Problem 34 — API review challenge

Review the following function conceptually:

```python
def calculate(data, mode, cache, precision, verbose):
    ...
```

A clearer Python 3.8-style API might be:

```python
def calculate(data, mode, /, *, cache=True, precision=3, verbose=False):
    ...
```

Reasons:

- `data` and `mode` are core operands.
- Their parameter names need not be a permanent public promise.
- Boolean and numeric configuration values are safer and clearer as keywords.
- Calls become self-documenting:

```python
calculate(records, "summary", cache=False, precision=6, verbose=True)
```

# 11. Additional Advanced Drills with Solutions

## Problem 35 — Inspect parameter kinds programmatically

Use `inspect.signature` to verify positional-only and keyword-only parameters.

In [41]:
def transform(data, scale, /, offset=0, *, clamp_to=None):
    result = data * scale + offset
    if clamp_to is not None:
        lower, upper = clamp_to
        result = min(max(result, lower), upper)
    return result


parameters = signature(transform).parameters
kinds = {name: parameter.kind.name for name, parameter in parameters.items()}

print(kinds)

assert kinds == {
    "data": "POSITIONAL_ONLY",
    "scale": "POSITIONAL_ONLY",
    "offset": "POSITIONAL_OR_KEYWORD",
    "clamp_to": "KEYWORD_ONLY",
}

{'data': 'POSITIONAL_ONLY', 'scale': 'POSITIONAL_ONLY', 'offset': 'POSITIONAL_OR_KEYWORD', 'clamp_to': 'KEYWORD_ONLY'}


## Problem 36 — Clear and reuse a cache in tests

Cache state can leak between tests. Demonstrate `cache_clear()` and verify fresh
statistics.

In [42]:
@lru_cache
def cube(value):
    return value ** 3


assert cube(3) == 27
assert cube(3) == 27
assert cube.cache_info().hits == 1

cube.cache_clear()
fresh = cube.cache_info()

print(f"{fresh=}")
assert fresh.hits == 0
assert fresh.misses == 0
assert fresh.currsize == 0

fresh=CacheInfo(hits=0, misses=0, maxsize=128, currsize=0)


## Problem 37 — Compare manual distance code with `math.dist`

The manual formula below contains a common indexing bug. Write the correct
version and compare it with `math.dist`.

In [43]:
a = (2, 5)
b = (8, 13)

# Correct manual formula:
manual_distance = math.sqrt(
    (b[0] - a[0]) ** 2 +
    (b[1] - a[1]) ** 2
)

library_distance = math.dist(a, b)

print(f"{manual_distance=}, {library_distance=}")
assert math.isclose(manual_distance, library_distance)
assert library_distance == 10.0

manual_distance=10.0, library_distance=10.0


Using `math.dist` avoids duplicated indexing logic, naturally supports more than
two dimensions, and states intent directly.

## Problem 38 — Reconstruct named-tuple defaults safely

Create a dictionary containing every field and either its default or a marker
showing that it is required.

In [44]:
REQUIRED = object()

def namedtuple_field_policy(record_type):
    defaults = record_type._field_defaults
    return {
        field: defaults.get(field, REQUIRED)
        for field in record_type._fields
    }


policy = namedtuple_field_policy(Connection)
readable_policy = {
    field: "<required>" if value is REQUIRED else value
    for field, value in policy.items()
}

print(readable_policy)

assert readable_policy == {
    "host": "<required>",
    "port": 443,
    "timeout": 5.0,
    "secure": True,
}

{'host': '<required>', 'port': 443, 'timeout': 5.0, 'secure': True}


## Problem 39 — Use a private sentinel instead of a literal marker

Do not use strings such as `"MISSING"` as sentinels because legitimate data can
equal that string.

In [45]:
_NOT_GIVEN = object()

def update_setting(settings, name, value=_NOT_GIVEN):
    if value is _NOT_GIVEN:
        return settings.get(name, _NOT_GIVEN)

    settings[name] = value
    return value


settings = {"mode": "MISSING"}

assert update_setting(settings, "mode") == "MISSING"
assert update_setting(settings, "unknown") is _NOT_GIVEN
assert update_setting(settings, "mode", None) is None
assert settings["mode"] is None

# 12. Final Checklist

Before using these features in production code, ask:

- **Positional-only:** Does hiding the parameter name improve API stability?
- **Keyword-only:** Would a positional call be ambiguous?
- **Debug f-string:** Could the expression expose secrets or trigger side effects?
- **`as_integer_ratio`:** Do you want the exact stored value or a human-friendly approximation?
- **`lru_cache`:** Are keys hashable, canonical, bounded appropriately, and safe from stale state?
- **`math.dist`:** Are dimensions validated?
- **`namedtuple` defaults:** Are any defaults mutable?
- **Reversed dict views:** Are you mutating the mapping during iteration?
- **`finally`:** Is cleanup free of `return`, `break`, `continue`, and exception suppression?
- **`is`:** Are you checking identity of a singleton/sentinel rather than value equality?

# End-to-End Verification

This final cell checks the most important reusable components created above.

In [46]:
assert final_price(50, 0.20, precision=2) == Decimal("60.00")
assert record_event("X", {}, event_type="metadata")["metadata"]["event_type"] == "metadata"
assert exact_fraction(Decimal("0.2")) == Fraction(1, 5)
assert moving_average([2, 4, 6], 2) == (3.0, 5.0)
assert nearest_point((0, 0), [(5, 5), (1, 1)])[0] == (1, 1)
assert Connection("host").port == 443
assert find_latest({"a": 1, "b": 2}, lambda key, value: value % 2 == 0) == ("b", 2)
assert lookup({"x": None}, "x") is None
assert math.isclose(score_route(((0, 0), (0, 3)), multiplier=2), 6.0)

print("All final verification checks passed.")

length=3.000, exact_multiplier=Fraction(2, 1), score=6.000
All final verification checks passed.
